In [3]:
import pandas as pd
import os
import json
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import spacy

def sub_scr(model: str):
    path = os.getcwd()
    if path.endswith("Utils"):
        path = os.path.dirname(os.path.dirname(path))


    model_path = os.path.join(
        path,
        "Models",
        f"{model}_NER",
        "output",
        "model-best"
    )
    trained_model = spacy.load(model_path)

    # =========================
    # Load dataset
    # =========================
    data_path = os.path.join(path, "Data", "raw", "Articles", "csv_format", "articles_dev.csv")
    test_set = pd.read_csv(data_path, sep='|')[["pmid", "title", "abstract"]]

    # =========================
    # Load linking model
    # =========================
    linking_model_path = os.path.join(
        path,
        "allenaiscibert_scivocab_uncased LAB15"
    )
    linking_model = SentenceTransformer(linking_model_path)

    # =========================
    # Load KB
    # =========================
    with open(os.path.join(path, "DataFile", "kb_synonyms_labels.pickle"), "rb") as f:
        kb_synonyms_labels = pickle.load(f)

    # =========================================================
    # BUILD ENTITY REPRESENTATION (FIXED: NO DEFINITIONS)
    # =========================================================
    entity_ids = []
    contexts = []

    for entity_id, aliases in kb_synonyms_labels.items():
        if len(aliases) == 0:
            continue

        # use canonical alias (first one)
        canonical = aliases[0]

        entity_ids.append(entity_id)
        contexts.append(canonical)

    # Encode entity embeddings (NORMALIZED)
    context_embeddings = linking_model.encode(
        contexts,
        normalize_embeddings=True,
        batch_size=64,
        show_progress_bar=True
    )

    # =========================
    # Linking function
    # =========================
    def link_entity(mention):
        mention_emb = linking_model.encode(
            mention,
            normalize_embeddings=True
        )

        scores = cosine_similarity([mention_emb], context_embeddings)[0]
        best_idx = np.argmax(scores)

        return entity_ids[best_idx], float(scores[best_idx])

    def id_to_uri(entity_id):
        return entity_id

    # =========================
    # Run NER + linking
    # =========================
    result = {}

    for idx, article in test_set.iterrows():
        print(f"Processing article {idx+1}/{len(test_set)}")

        pmid = int(article["pmid"])
        title = article["title"]
        abstract = article["abstract"]

        result[pmid] = {"entities": []}

        # NOTE: assumes trained NER model already exists
        nlp_title = trained_model(title)
        nlp_abstract = trained_model(abstract)

        # Title entities
        for ent in nlp_title.ents:
            entity_id, score = link_entity(ent.text)

            result[pmid]["entities"].append({
                "start_idx": ent.start_char,
                "end_idx": ent.end_char,
                "location": "title",
                "text_span": ent.text,
                "label": ent.label_,
                "uri": id_to_uri(entity_id)
            })

        # Abstract entities
        for ent in nlp_abstract.ents:
            entity_id, score = link_entity(ent.text)

            result[pmid]["entities"].append({
                "start_idx": ent.start_char,
                "end_idx": ent.end_char,
                "location": "abstract",
                "text_span": ent.text,
                "label": ent.label_,
                "uri": id_to_uri(entity_id)
            })

    # =========================
    # Save output
    # =========================
    output_path = os.path.join(path, "Output", "submission_NER.json")

    with open(output_path, "w") as f:
        json.dump(result, f, indent=4)

    return result


# Run submission
NER = sub_scr("SciBERT")

C:\Users\ohs\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\spacy\util.py:971: UserWarning: [W095] Model 'en_pipeline' (0.0.0) was trained with spaCy v3.7.5 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
Batches: 100%|██████████| 45/45 [00:15<00:00,  2.82it/s]


Processing article 1/80
Processing article 2/80
Processing article 3/80
Processing article 4/80
Processing article 5/80
Processing article 6/80
Processing article 7/80
Processing article 8/80
Processing article 9/80
Processing article 10/80
Processing article 11/80
Processing article 12/80
Processing article 13/80
Processing article 14/80
Processing article 15/80
Processing article 16/80
Processing article 17/80
Processing article 18/80
Processing article 19/80
Processing article 20/80
Processing article 21/80
Processing article 22/80
Processing article 23/80
Processing article 24/80
Processing article 25/80
Processing article 26/80
Processing article 27/80
Processing article 28/80
Processing article 29/80
Processing article 30/80
Processing article 31/80
Processing article 32/80
Processing article 33/80
Processing article 34/80
Processing article 35/80
Processing article 36/80
Processing article 37/80
Processing article 38/80
Processing article 39/80
Processing article 40/80
Processin